In [1]:
pip install gradio pandas matplotlib

Note: you may need to restart the kernel to use updated packages.


In [2]:
pip install gradio typer click

Note: you may need to restart the kernel to use updated packages.


In [3]:
import io
from pathlib import Path

import gradio as gr
import matplotlib.pyplot as plt
import pandas as pd

In [4]:
# These imports assume you place this app in the same project folder
# as the Python modules created for your project.
DATA_PATH = Path("business_ready_data.csv")
PROJECT_DATA_PATH = Path("Telco_customer_churn.xlsx")
HIGH_RISK = 0.60
HIGH_VALUE = 0.70

In [5]:
def get_project_data() -> pd.DataFrame:
    """
    Strictly uses the project's own processed data only.
    Priority:
    1) business_ready_data.csv
    2) final_scored_data.csv
    3) modeled_churn_data.csv
    """
    if DATA_PATH.exists():
        return pd.read_csv(DATA_PATH)

    fallback_paths = [Path("final_scored_data.csv"), Path("modeled_churn_data.csv"), Path("segmented_data.csv")]
    for path in fallback_paths:
        if path.exists():
            return pd.read_csv(path)

    raise FileNotFoundError(
        "Could not find business_ready_data.csv or other processed project CSV files in the project folder."
    )


def make_template_file() -> str:
    # Intentionally disabled because this version is project-data only.
    path = Path("project_data_only.txt")
    path.write_text(
        "This app is locked to the Telco Customer Churn project dataset and pipeline."
        "Use WA_Fn-UseC_-Telco-Customer-Churn.csv or business_ready_data.csv from this project."
        ,
        encoding="utf-8",
    )
    return str(path)


In [6]:
def get_overview():
    df = get_project_data()

    total_customers = len(df)
    churn_rate = float(df["Churn"].mean())
    high_risk = int((df["churn_probability"] >= HIGH_RISK).sum())
    revenue_at_risk = float(df.loc[df["churn_probability"] >= HIGH_RISK, "Total Charges"].sum())
    avg_value = float(df["value_score"].mean())

    fig_scatter, ax = plt.subplots(figsize=(8, 5))
    ax.scatter(df["churn_probability"], df["value_score"], alpha=0.5)
    ax.axvline(HIGH_RISK, linestyle="--")
    ax.axhline(HIGH_VALUE, linestyle="--")
    ax.set_xlabel("Churn Probability")
    ax.set_ylabel("Value Score")
    ax.set_title("Customer Risk vs Value")

    fig_category, ax2 = plt.subplots(figsize=(8, 4))
    df["customer_category"].value_counts().plot(kind="barh", ax=ax2)
    ax2.set_xlabel("Customers")
    ax2.set_ylabel("")
    ax2.set_title("Customer Category Distribution")

    fig_action, ax3 = plt.subplots(figsize=(8, 4))
    df["recommended_action"].value_counts().plot(kind="barh", ax=ax3)
    ax3.set_xlabel("Customers")
    ax3.set_ylabel("")
    ax3.set_title("Recommended Actions")

    fig_prob, ax4 = plt.subplots(figsize=(8, 4))
    ax4.hist(df["churn_probability"], bins=25, alpha=0.8)
    ax4.axvline(HIGH_RISK, linestyle="--")
    ax4.set_xlabel("Churn Probability")
    ax4.set_ylabel("Count")
    ax4.set_title("Churn Probability Distribution")

    summary_md = f"""
### Overview
- **Total Customers:** {total_customers:,}
- **Churn Rate:** {churn_rate:.1%}
- **High-Risk Customers:** {high_risk:,}
- **Average Value Score:** {avg_value:.2f}
- **Revenue at Risk:** ${revenue_at_risk:,.0f}
"""

    note = (
        "This app is locked to your processed Telco project data only. "
        "It reads your CSV outputs directly from the project folder."
    )

    customer_choices = df["customer_id_display"].tolist() if "customer_id_display" in df.columns else [str(i) for i in df.index]
    if "customer_id_display" not in df.columns:
        df = df.copy()
        df["customer_id_display"] = customer_choices

    return summary_md, fig_scatter, fig_category, fig_action, fig_prob, note, df, gr.update(choices=customer_choices, value=customer_choices[0] if customer_choices else None)


In [7]:
def get_customer_view(df_state, customer_id):
    if df_state is None:
        raise gr.Error("Load the project data first from the Overview tab.")

    df = pd.DataFrame(df_state)
    row = df.loc[df["customer_id_display"] == customer_id]
    if row.empty:
        raise gr.Error("Customer not found.")
    row = row.iloc[0]

    info = pd.DataFrame({
        "Field": [
            "Customer ID",
            "Segment",
            "Customer Category",
            "Churn Probability",
            "Value Score",
            "Recommended Action",
        ],
        "Value": [
            row.get("customer_id_display", "N/A"),
            row.get("segment", "N/A"),
            row.get("customer_category", "N/A"),
            f"{row.get('churn_probability', 0):.1%}",
            f"{row.get('value_score', 0):.2f}",
            row.get("recommended_action", "N/A"),
        ]
    })

    snapshot_cols = [
        c for c in [
            "Gender",
            "Senior Citizen",
            "Partner",
            "Dependents",
            "Contract",
            "Payment Method",
            "Monthly Charges",
            "Total Charges",
            "service_count",
            "engagement_score",
        ] if c in df.columns
    ]
    snapshot = row[snapshot_cols].to_frame("Value").reset_index()
    snapshot.columns = ["Feature", "Value"]
    return info, snapshot


In [12]:
project_note_file = gr.File()
def get_business_view(df_state):
    if df_state is None:
        raise gr.Error("Load the project data first from the Overview tab.")

    df = pd.DataFrame(df_state)
    category_df = df["customer_category"].value_counts().reset_index()
    category_df.columns = ["Customer Category", "Count"]

    actions_df = df["recommended_action"].value_counts().reset_index()
    actions_df.columns = ["Recommended Action", "Count"]

    top_cols = [
        c for c in [
            "customer_id_display",
            "segment",
            "churn_probability",
            "value_score",
            "recommended_action",
            "Total Charges",
        ] if c in df.columns
    ]
    top_risk = df.sort_values(["churn_probability", "value_score"], ascending=[False, False]).head(10)[top_cols]

    return category_df, actions_df, top_risk


def get_model_insights(df_state):
    if df_state is None:
        raise gr.Error("Load the project data first from the Overview tab.")

    df = pd.DataFrame(df_state)

    cols = [c for c in ["Contract", "Monthly Charges", "Total Charges", "Tenure Months", "service_count", "engagement_score"] if c in df.columns]
    stats = df[cols].describe(include="all").transpose() if cols else pd.DataFrame()

    fig, ax = plt.subplots(figsize=(8, 4))
    ax.hist(df.loc[df["Churn"] == 0, "churn_probability"], bins=25, alpha=0.6, label="Stayed")
    ax.hist(df.loc[df["Churn"] == 1, "churn_probability"], bins=25, alpha=0.6, label="Churned")
    ax.set_xlabel("Churn Probability")
    ax.set_ylabel("Count")
    ax.set_title("Probability Distribution by Actual Churn")
    ax.legend()

    return stats, fig


def export_results(df_state):
    if df_state is None:
        raise gr.Error("Load the project data first from the Overview tab.")

    df = pd.DataFrame(df_state)
    out_path = Path("customer_intelligence_results.csv")
    df.to_csv(out_path, index=False)
    return str(out_path)


with gr.Blocks(title="Customer Intelligence & Retention System") as demo:
    df_state = gr.State(value=None)

    gr.Markdown("# AI Customer Intelligence & Retention System")
    gr.Markdown("This Gradio app is locked to your Telco Customer Churn project data and your own Python pipeline modules.")

    with gr.Row():
        load_btn = gr.Button("Load Project Data")

    project_note_btn.click(fn=make_template_file, outputs=project_note_file)

    with gr.Tab("Overview"):
        overview_md = gr.Markdown()
        overview_plot = gr.Plot(label="Risk vs Value")
        overview_plot_2 = gr.Plot(label="Customer Categories")
        overview_plot_3 = gr.Plot(label="Recommended Actions")
        overview_plot_4 = gr.Plot(label="Probability Distribution")
        compatibility_box = gr.Textbox(label="Project Mode", lines=3)
        customer_dropdown = gr.Dropdown(label="Customer", choices=[])

        load_btn.click(
            fn=get_overview,
            inputs=None,
            outputs=[overview_md, overview_plot, overview_plot_2, overview_plot_3, overview_plot_4, compatibility_box, df_state, customer_dropdown],
        )

    with gr.Tab("Customer Lookup"):
        customer_btn = gr.Button("View Customer")
        customer_info = gr.Dataframe(label="Customer Summary", interactive=False)
        customer_snapshot = gr.Dataframe(label="Customer Snapshot", interactive=False)

        customer_btn.click(
            fn=get_customer_view,
            inputs=[df_state, customer_dropdown],
            outputs=[customer_info, customer_snapshot],
        )

    with gr.Tab("Business View"):
        business_btn = gr.Button("Load Business View")
        category_table = gr.Dataframe(label="Customer Category Distribution", interactive=False)
        action_table = gr.Dataframe(label="Recommended Actions Distribution", interactive=False)
        top_risk_table = gr.Dataframe(label="Top Customers To Save", interactive=False)

        business_btn.click(
            fn=get_business_view,
            inputs=df_state,
            outputs=[category_table, action_table, top_risk_table],
        )

    with gr.Tab("Model Insights"):
        insights_btn = gr.Button("Load Model Insights")
        stats_table = gr.Dataframe(label="Feature Snapshot", interactive=False)
        probs_plot = gr.Plot()

        insights_btn.click(
            fn=get_model_insights,
            inputs=df_state,
            outputs=[stats_table, probs_plot],
        )

    with gr.Tab("Export"):
        export_btn = gr.Button("Prepare CSV Export")
        export_file = gr.File(label="Download Results")

        export_btn.click(fn=export_results, inputs=df_state, outputs=export_file)


demo.launch(share=True)


* Running on local URL:  http://127.0.0.1:7861
* Running on public URL: https://8e6b380485d67fa243.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


## Business Insights

- The churn rate is approximately 26%, meaning that more than 1 in 4 customers leave the company.
- Around 1,146 customers are identified as high-risk, representing a critical group for retention efforts.
- The estimated revenue at risk is $931,732, demonstrating the significant financial impact of customer churn.
- The average customer value score is relatively low (0.30), indicating that only a subset of customers contributes high value.

These findings highlight the importance of prioritizing high-value customers at risk of churn to maximize retention impact.